STS B: https://huggingface.co/datasets/PhilipMay/stsb_multi_mt 

In [ ]:
# Load data
import pandas as pd

# STS B
splits = {'train': 'en/train-00000-of-00001.parquet', 'test': 'en/test-00000-of-00001.parquet', 'dev': 'en/dev-00000-of-00001.parquet'}
STSB_train = pd.read_parquet("hf://datasets/PhilipMay/stsb_multi_mt/" + splits["train"])
STSB_test = pd.read_parquet("hf://datasets/PhilipMay/stsb_multi_mt/" + splits["test"])
STSB_dev = pd.read_parquet("hf://datasets/PhilipMay/stsb_multi_mt/" + splits["dev"])

Here, we filter the dataframe

In [ ]:
import numpy as np

# Turn to right type
STSB_train["similarity_score"] = pd.to_numeric(STSB_train["similarity_score"])

# Remove similarity scores in [1,4]
filtered = STSB_train[
    (STSB_train["similarity_score"] <= 1) | 
    (STSB_train["similarity_score"] >= 4)
]

# Add binary "same" column
filtered["same"] = np.where(filtered["similarity_score"] > 3, 1, 0)

Lemmatize sentences

In [ ]:
import spacy

tokenizer = spacy.load('en_core_web_sm')

# Tokenize STSB
filtered["lemma1"] = [
    [token.lemma_ for token in doc]
    for doc in tokenizer.pipe(filtered["sentence1"].str.lower())
]
filtered["lemma2"] = [
    [token.lemma_ for token in doc]
    for doc in tokenizer.pipe(filtered["sentence2"].str.lower())
]

Here we define a function to calculate the lexical overlap

In [ ]:
def jaccard(a,b):
    return len(a & b) / len(a | b)

filtered["lex_overlap"] = filtered.apply(
    lambda row: jaccard(set(row["lemma1"]), set(row["lemma2"])),
    axis=1
)

In [ ]:
from transformers import AutoTokenizer, GPT2Tokenizer
# First GPT-2 Tokenization
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

filtered["GPT_ID1"] = filtered["sentence1"].apply(
    lambda s: tokenizer.encode(s)
)

filtered["GPT_ID2"] = filtered["sentence2"].apply(
    lambda s: tokenizer.encode(s)
)

# Now the E5 Tokenization
tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-small-v2")

filtered["E5_ID1"] = filtered["sentence1"].apply(
    lambda s: tokenizer.encode(s)
)

filtered["E5_ID2"] = filtered["sentence2"].apply(
    lambda s: tokenizer.encode(s)
)


In [ ]:
filtered = filtered.drop(columns=["lemma1", "lemma2"])
filtered.to_csv("Data.csv", sep=";", index=False)